In [10]:
def to_twos_complement_hex(val, bits):
    """Converts a signed integer to a zero-padded hex string."""
    mask = (1 << bits) - 1
    hex_chars = (bits + 3) // 4  
    return f"0x{(val & mask):0{hex_chars}X}"

def from_twos_complement_hex(hex_val, bits):
    """Converts a hardware hex literal into a signed Python integer."""
    if hex_val & (1 << (bits - 1)):
        hex_val -= (1 << bits)
    return hex_val

def hardware_add_extended(a, b, input_width):
    """
    Simulates hardware addition where the output width grows by 1 bit 
    to completely prevent wrap-around overflow.
    """
    output_width = input_width + 1
    
    # 1. Standard addition
    raw_sum = a + b
    
    # 2. Simulate storing the sum in an extended 17-bit register
    mask_n = (1 << output_width) - 1
    sum_hw = raw_sum & mask_n
    
    # 3. Convert the raw hardware bits back to a signed Python integer
    if sum_hw & (1 << (output_width - 1)):
        return sum_hw - (1 << output_width)
    return sum_hw

def run_adder_simulation():
    # Scale factor for 15 fractional bits (2^15)
    SCALE = 32768.0
    INPUT_BITS = 16
    OUTPUT_BITS = 17 # Q2.15 format

    # Hex inputs extracted exactly from your Verilog testbench
    test_cases_hex = [
        (0x0180, 0x0280),   # TC1: 0.0117 + 0.0195
        (0xFE80, 0x0080),   # TC2: -0.0117 + 0.0039
        (0x7FFF, 0x7FFF),   # TC3: Max positive
        (0x8000, 0x8000),   # TC4: Max negative
        (0x03E8, 0xFC18)    # TC5: Zero Crossing
    ]

    print("Starting Python Q2.15 Extended Adder Simulation...")
    print("=========================================================================================")

    for idx, (a_hex_in, b_hex_in) in enumerate(test_cases_hex, 1):
        # Decode hex to 16-bit signed integer
        a = from_twos_complement_hex(a_hex_in, INPUT_BITS)
        b = from_twos_complement_hex(b_hex_in, INPUT_BITS)

        # Emulate the Verilog 17-bit hardware addition
        sum_val = hardware_add_extended(a, b, input_width=INPUT_BITS)

        # Convert to floating point for readability
        a_float = a / SCALE
        b_float = b / SCALE
        sum_float = sum_val / SCALE

        # Encode back to hardware hex for output verification
        # Inputs are passed as 16-bit, sum is passed as 17-bit
        a_hex_out = to_twos_complement_hex(a, INPUT_BITS)
        b_hex_out = to_twos_complement_hex(b, INPUT_BITS)
        sum_hex_out = to_twos_complement_hex(sum_val, OUTPUT_BITS)

        # Print results. Notice the sum requires an extra hex character (5 chars)
        print(f"Time={idx * 15} | a = {a_float:8.4f} ({a_hex_out}) | b = {b_float:8.4f} ({b_hex_out}) | sum = {sum_float:8.4f} ({sum_hex_out})")

if __name__ == "__main__":
    run_adder_simulation()

Starting Python Q2.15 Extended Adder Simulation...
Time=15 | a =   0.0117 (0x0180) | b =   0.0195 (0x0280) | sum =   0.0312 (0x00400)
Time=30 | a =  -0.0117 (0xFE80) | b =   0.0039 (0x0080) | sum =  -0.0078 (0x1FF00)
Time=45 | a =   1.0000 (0x7FFF) | b =   1.0000 (0x7FFF) | sum =   1.9999 (0x0FFFE)
Time=60 | a =  -1.0000 (0x8000) | b =  -1.0000 (0x8000) | sum =  -2.0000 (0x10000)
Time=75 | a =   0.0305 (0x03E8) | b =  -0.0305 (0xFC18) | sum =   0.0000 (0x00000)


In [9]:
def to_twos_complement_hex(val, bits):
    """Converts a signed integer to a zero-padded hex string."""
    mask = (1 << bits) - 1
    hex_chars = (bits + 3) // 4  
    return f"0x{(val & mask):0{hex_chars}X}"

def from_twos_complement_hex(hex_val, bits):
    """Converts a hardware hex literal into a signed Python integer."""
    if hex_val & (1 << (bits - 1)):
        hex_val -= (1 << bits)
    return hex_val

def hardware_multiply(a, b, int_width, frac_width):
    """
    Simulates the exact bit-level truncation and saturation logic of the Verilog multiplier.
    """
    total_width = int_width + frac_width
    
    # Calculate dynamic Min and Max saturation values (matching Verilog logic)
    min_val = -(1 << (total_width - 1))        # e.g., -32768 for 16-bit
    max_val = (1 << (total_width - 1)) - 1     # e.g., 32767 for 16-bit
    
    # ---------------------------------------------------------
    # Saturation Logic for the -1.0 * -1.0 Edge Case
    # ---------------------------------------------------------
    # If both inputs are exactly the minimum negative value, clamp the output 
    # to the maximum positive value.
    if a == min_val and b == min_val:
        return max_val
        
    # 1. Full precision hardware multiplication
    full_product = a * b
    
    # 2. Simulate a hardware register of 2*N bits (32 bits for Q1.15)
    mask_2n = (1 << (2 * total_width)) - 1
    hw_full_product = full_product & mask_2n
    
    # 3. Apply the part-select: Shift right by FRAC_WIDTH and mask to TOTAL_WIDTH
    mask_n = (1 << total_width) - 1
    prod_raw = (hw_full_product >> frac_width) & mask_n
    
    # 4. Convert the raw hardware bits back to a signed Python integer
    if prod_raw & (1 << (total_width - 1)):
        return prod_raw - (1 << total_width)
    return prod_raw

def run_multiplier_simulation():
    # Scale factor for Q1.15 (2^15)
    SCALE = 32768.0
    TOTAL_BITS = 16

    # Test cases mapped exactly to the Verilog TB
    test_cases_hex = [
        (0x4000, 0x4000),   # TC1:  0.5 * 0.5
        (0xC000, 0x2000),   # TC2: -0.5 * 0.25
        (0x1000, 0x1000),   # TC3:  0.125 * 0.125
        (0xA000, 0xA000),   # TC4: -0.75 * -0.75
        (0x8000, 0x8000)    # TC5: -1.0 * -1.0 (Now correctly saturating)
    ]

    print("Starting Python Q1.15 Multiplier Simulation (With Saturation)...")
    print("=" * 75)

    for idx, (a_hex_in, b_hex_in) in enumerate(test_cases_hex, 1):
        # Decode hex to signed integer
        a = from_twos_complement_hex(a_hex_in, TOTAL_BITS)
        b = from_twos_complement_hex(b_hex_in, TOTAL_BITS)

        # Emulate the Verilog hardware logic with Q1.15 parameters
        prod = hardware_multiply(a, b, int_width=1, frac_width=15)

        # Convert to floating point for readability
        a_float = a / SCALE
        b_float = b / SCALE
        prod_float = prod / SCALE

        # Encode back to hardware hex for output verification
        a_hex_out = to_twos_complement_hex(a, TOTAL_BITS)
        b_hex_out = to_twos_complement_hex(b, TOTAL_BITS)
        prod_hex_out = to_twos_complement_hex(prod, TOTAL_BITS)

        # Print results mimicking $monitor
        print(f"Test Case {idx}:")
        print(f"  a    = {a_float:8.4f} ({a_hex_out})")
        print(f"  b    = {b_float:8.4f} ({b_hex_out})")
        print(f"  prod = {prod_float:8.4f} ({prod_hex_out})")
        print("-" * 75)

if __name__ == "__main__":
    run_multiplier_simulation()

Starting Python Q1.15 Multiplier Simulation (With Saturation)...
Test Case 1:
  a    =   0.5000 (0x4000)
  b    =   0.5000 (0x4000)
  prod =   0.2500 (0x2000)
---------------------------------------------------------------------------
Test Case 2:
  a    =  -0.5000 (0xC000)
  b    =   0.2500 (0x2000)
  prod =  -0.1250 (0xF000)
---------------------------------------------------------------------------
Test Case 3:
  a    =   0.1250 (0x1000)
  b    =   0.1250 (0x1000)
  prod =   0.0156 (0x0200)
---------------------------------------------------------------------------
Test Case 4:
  a    =  -0.7500 (0xA000)
  b    =  -0.7500 (0xA000)
  prod =   0.5625 (0x4800)
---------------------------------------------------------------------------
Test Case 5:
  a    =  -1.0000 (0x8000)
  b    =  -1.0000 (0x8000)
  prod =   1.0000 (0x7FFF)
---------------------------------------------------------------------------


COMPLEX MULT

In [11]:
def to_twos_complement_hex(val, bits):
    """Converts a signed integer to a zero-padded hex string."""
    mask = (1 << bits) - 1
    hex_chars = (bits + 3) // 4  
    return f"0x{(val & mask):0{hex_chars}X}"

def from_twos_complement_hex(hex_val, bits):
    """Converts a hardware hex literal into a signed Python integer."""
    if hex_val & (1 << (bits - 1)):
        hex_val -= (1 << bits)
    return hex_val

def hardware_multiply(a, b, total_width=18, frac_width=15):
    """Simulates the 18-bit fixed_point_multiplier."""
    full_product = a * b
    mask_2n = (1 << (2 * total_width)) - 1
    hw_full_product = full_product & mask_2n
    
    mask_n = (1 << total_width) - 1
    prod_raw = (hw_full_product >> frac_width) & mask_n
    
    if prod_raw & (1 << (total_width - 1)):
        return prod_raw - (1 << total_width)
    return prod_raw

def hardware_slice_17_to_2(val, in_width=18):
    """
    Simulates: assign rr = PRErr[TOTAL_WIDTH-1 : 2];
    Extracts 16 bits, effectively dividing by 4 and dropping top integer bits.
    """
    # Force to unsigned hardware representation
    hw_val = val & ((1 << in_width) - 1)
    
    # Shift right by 2 (extracts [17:2])
    sliced = hw_val >> 2
    
    # Result is 16 bits. Convert back to signed Python integer.
    if sliced & (1 << 15):
        return sliced - (1 << 16)
    return sliced

def hardware_negate(val, bits=16):
    """Simulates hardware 2s complement negation (e.g., -ii)."""
    hw_neg = (-val) & ((1 << bits) - 1)
    if hw_neg & (1 << (bits - 1)):
        return hw_neg - (1 << bits)
    return hw_neg

def hardware_add(a, b, in_width=16):
    """Simulates the adder outputting a 17-bit wire (IN_WIDTH + 1)."""
    raw_sum = a + b
    out_width = in_width + 1
    
    sum_hw = raw_sum & ((1 << out_width) - 1)
    if sum_hw & (1 << (out_width - 1)):
        return sum_hw - (1 << out_width)
    return sum_hw

def hardware_truncate(val, in_width=17, out_width=16):
    """Simulates: prod_r <= preProd_r[TOTAL_WIDTH-3:0] (16 bits)"""
    trunc = val & ((1 << out_width) - 1)
    if trunc & (1 << (out_width - 1)):
        return trunc - (1 << out_width)
    return trunc

def run_complex_multiplier_simulation():
    SCALE = 32768.0
    IN_BITS = 18   # Q3.15
    OUT_BITS = 16  # Q1.15

    # Test cases mapped exactly to the Verilog TB
    test_cases = [
        # (a_r, a_i, b_r, b_i)
        (0x10000, 0x00000, 0x0C000, 0x00000), # TC1:  2.0 * 1.5
        (0x08000, 0x08000, 0x08000, 0x08000), # TC2: (1+1j) * (1+1j)
        (0x0C000, 0x00000, 0x34000, 0x00000)  # TC3:  1.5 * -1.5
    ]

    print("Starting Python 2-Cycle Complex Multiplier Simulation...")
    print("=========================================================================================")

    for idx, (ar_hex, ai_hex, br_hex, bi_hex) in enumerate(test_cases, 1):
        # 0. Decode hex to signed integers
        a_r = from_twos_complement_hex(ar_hex, IN_BITS)
        a_i = from_twos_complement_hex(ai_hex, IN_BITS)
        b_r = from_twos_complement_hex(br_hex, IN_BITS)
        b_i = from_twos_complement_hex(bi_hex, IN_BITS)

        # 1. Hardware Multipliers (18-bit)
        PRErr = hardware_multiply(a_r, b_r)
        PREri = hardware_multiply(a_r, b_i)
        PREir = hardware_multiply(a_i, b_r)
        PREii = hardware_multiply(a_i, b_i)

        # 2. Hardware Slicing (18-bit to 16-bit)
        rr = hardware_slice_17_to_2(PRErr)
        ri = hardware_slice_17_to_2(PREri)
        ir = hardware_slice_17_to_2(PREir)
        ii = hardware_slice_17_to_2(PREii)

        # 3. Hardware Adders (16-bit to 17-bit)
        preProd_r = hardware_add(rr, hardware_negate(ii, OUT_BITS))
        preProd_i = hardware_add(ri, ir)

        # 4. Hardware Output Register Truncation (17-bit to 16-bit)
        prod_r = hardware_truncate(preProd_r)
        prod_i = hardware_truncate(preProd_i)

        # 5. Format and Print Results
        print(f"Test Case {idx}:")
        print(f"  A = {a_r/SCALE:6.3f} + j{a_i/SCALE:6.3f}")
        print(f"  B = {b_r/SCALE:6.3f} + j{b_i/SCALE:6.3f}")
        print(f"  Scaled Prod (A*B/4) = {prod_r/SCALE:6.3f} + j{prod_i/SCALE:6.3f}")
        print(f"  Hex Out: Real={to_twos_complement_hex(prod_r, OUT_BITS)}, Imag={to_twos_complement_hex(prod_i, OUT_BITS)}")
        print("-" * 89)

if __name__ == "__main__":
    run_complex_multiplier_simulation()

Starting Python 2-Cycle Complex Multiplier Simulation...
Test Case 1:
  A =  2.000 + j 0.000
  B =  1.500 + j 0.000
  Scaled Prod (A*B/4) =  0.750 + j 0.000
  Hex Out: Real=0x6000, Imag=0x0000
-----------------------------------------------------------------------------------------
Test Case 2:
  A =  1.000 + j 1.000
  B =  1.000 + j 1.000
  Scaled Prod (A*B/4) =  0.000 + j 0.500
  Hex Out: Real=0x0000, Imag=0x4000
-----------------------------------------------------------------------------------------
Test Case 3:
  A =  1.500 + j 0.000
  B = -1.500 + j 0.000
  Scaled Prod (A*B/4) = -0.562 + j 0.000
  Hex Out: Real=0xB800, Imag=0x0000
-----------------------------------------------------------------------------------------


BF4 complex

In [2]:
def radix4_butterfly_true_math(a, b, c, d, w1, w2, w3):
    """
    Computes the exact mathematical Radix-4 DIF butterfly using 
    Python's native complex numbers. No fixed-point scaling.
    """
    # Stage 1: Additions
    s0 = a + c
    s1 = a - c
    s2 = b + d
    s3 = b - d
    
    # Stage 2: Cross Additions (with -j and +j)
    # Note: In Python, '1j' is the imaginary unit.
    A_pre = s0 + s2
    B_pre = s1 - (1j * s3)
    C_pre = s0 - s2
    D_pre = s1 + (1j * s3)
    
    # Stage 3: Twiddle Multiplication
    A = A_pre           # A requires no twiddle factor
    B = B_pre * w1
    C = C_pre * w2
    D = D_pre * w3
    
    return A, B, C, D

def run_true_math_simulation():
    # Test cases exactly matching the Verilog TB
    # Format: (a, b, c, d) - all are complex numbers
    test_cases = [
        # TC1: DC Input (All inputs are 0.5 + 0j)
        (complex(0.5, 0), complex(0.5, 0), complex(0.5, 0), complex(0.5, 0)),
        
        # TC2: Impulse Input (A = 0.5, Rest = 0)
        (complex(0.5, 0), complex(0.0, 0), complex(0.0, 0), complex(0.0, 0))
    ]

    # Standard trivial twiddles for these test cases (W^0 = 1.0 + 0j)
    w1 = complex(1.0, 0.0)
    w2 = complex(1.0, 0.0)
    w3 = complex(1.0, 0.0)

    print("Starting Pure Floating-Point Radix-4 Simulation...")
    print("===================================================")

    for idx, (a, b, c, d) in enumerate(test_cases, 1):
        # Calculate true mathematical output
        A, B, C, D = radix4_butterfly_true_math(a, b, c, d, w1, w2, w3)
        
        print(f"Test Case {idx}:")
        print("  --- Inputs ---")
        print(f"  a = {a.real:8.4f} + j{a.imag:8.4f}")
        print(f"  b = {b.real:8.4f} + j{b.imag:8.4f}")
        print(f"  c = {c.real:8.4f} + j{c.imag:8.4f}")
        print(f"  d = {d.real:8.4f} + j{d.imag:8.4f}")
        
        # print("\n  --- True Mathematical Output (Unscaled) ---")
        # print(f"  A = {A.real:8.4f} + j{A.imag:8.4f}")
        # print(f"  B = {B.real:8.4f} + j{B.imag:8.4f}")
        # print(f"  C = {C.real:8.4f} + j{C.imag:8.4f}")
        # print(f"  D = {D.real:8.4f} + j{D.imag:8.4f}")
        
        print("\n  --- Expected Hardware Output (Scaled by /4) ---")
        print(f"  A = {A.real/4:8.4f} + j{A.imag/4:8.4f}")
        print(f"  B = {B.real/4:8.4f} + j{B.imag/4:8.4f}")
        print(f"  C = {C.real/4:8.4f} + j{C.imag/4:8.4f}")
        print(f"  D = {D.real/4:8.4f} + j{D.imag/4:8.4f}")
        print("-" * 51)

if __name__ == "__main__":
    run_true_math_simulation()

Starting Pure Floating-Point Radix-4 Simulation...
Test Case 1:
  --- Inputs ---
  a =   0.5000 + j  0.0000
  b =   0.5000 + j  0.0000
  c =   0.5000 + j  0.0000
  d =   0.5000 + j  0.0000

  --- Expected Hardware Output (Scaled by /4) ---
  A =   0.5000 + j  0.0000
  B =   0.0000 + j  0.0000
  C =   0.0000 + j  0.0000
  D =   0.0000 + j  0.0000
---------------------------------------------------
Test Case 2:
  --- Inputs ---
  a =   0.5000 + j  0.0000
  b =   0.0000 + j  0.0000
  c =   0.0000 + j  0.0000
  d =   0.0000 + j  0.0000

  --- Expected Hardware Output (Scaled by /4) ---
  A =   0.1250 + j  0.0000
  B =   0.1250 + j  0.0000
  C =   0.1250 + j  0.0000
  D =   0.1250 + j  0.0000
---------------------------------------------------


BF4 real

In [4]:
def radix4_real_input_butterfly(a, b, c, d, w1, w2, w3):
    """
    Computes the exact mathematical Radix-4 DIF butterfly 
    assuming purely real inputs (a, b, c, d).
    """
    # Stage 1: Standard Additions
    s0 = a + c
    s1 = a - c
    s2 = b + d
    s3 = b - d
    
    # Stage 2: Cross Additions (Forming Complex Numbers)
    A_pre = complex(s0 + s2, 0.0)
    C_pre = complex(s0 - s2, 0.0)
    
    # -j*s3 means real=0, imag=-s3. Added to s1 (real).
    B_pre = complex(s1, -s3)
    
    # +j*s3 means real=0, imag=s3. Added to s1 (real).
    D_pre = complex(s1, s3)
    
    # Stage 3: Twiddle Multiplication
    A = A_pre          # A requires no twiddle factor
    B = B_pre * w1
    C = C_pre * w2
    D = D_pre * w3
    
    return A, B, C, D

def run_real_math_simulation():
    # Test cases exactly matching your tb_BF4real.v
    # Format: (a, b, c, d) - purely real floats
    test_cases = [
        # TC1: DC Input 
        (0.5, 0.5, 0.5, 0.5),
        
        # TC2: Impulse Input 
        (0.5, 0.0, 0.0, 0.0),
        
        # TC3: Alternating Pattern
        (0.5, -0.5, 0.5, -0.5)
    ]

    # Standard trivial twiddles for these test cases (W^0 = 1.0 + j0.0)
    w1 = complex(1.0, 0.0)
    w2 = complex(1.0, 0.0)
    w3 = complex(1.0, 0.0)

    print("Starting Pure Floating-Point Real-Input BF4 Simulation...")
    print("=========================================================")

    for idx, (a, b, c, d) in enumerate(test_cases, 1):
        # Calculate true mathematical output
        A, B, C, D = radix4_real_input_butterfly(a, b, c, d, w1, w2, w3)
        
        print(f"Test Case {idx}:")
        print("  --- Inputs ---")
        print(f"  a = {a:8.4f}")
        print(f"  b = {b:8.4f}")
        print(f"  c = {c:8.4f}")
        print(f"  d = {d:8.4f}")
        
        # print("\n  --- True Mathematical Output (Unscaled) ---")
        # print(f"  A = {A.real:8.4f} + j{A.imag:8.4f}")
        # print(f"  B = {B.real:8.4f} + j{B.imag:8.4f}")
        # print(f"  C = {C.real:8.4f} + j{C.imag:8.4f}")
        # print(f"  D = {D.real:8.4f} + j{D.imag:8.4f}")
        
        print("\n  --- Expected Hardware Output (Scaled by /4) ---")
        print(f"  A = {A.real/4:8.4f} + j{A.imag/4:8.4f}")
        print(f"  B = {B.real/4:8.4f} + j{B.imag/4:8.4f}")
        print(f"  C = {C.real/4:8.4f} + j{C.imag/4:8.4f}")
        print(f"  D = {D.real/4:8.4f} + j{D.imag/4:8.4f}")
        print("-" * 57)

if __name__ == "__main__":
    run_real_math_simulation()

Starting Pure Floating-Point Real-Input BF4 Simulation...
Test Case 1:
  --- Inputs ---
  a =   0.5000
  b =   0.5000
  c =   0.5000
  d =   0.5000

  --- Expected Hardware Output (Scaled by /4) ---
  A =   0.5000 + j  0.0000
  B =   0.0000 + j  0.0000
  C =   0.0000 + j  0.0000
  D =   0.0000 + j  0.0000
---------------------------------------------------------
Test Case 2:
  --- Inputs ---
  a =   0.5000
  b =   0.0000
  c =   0.0000
  d =   0.0000

  --- Expected Hardware Output (Scaled by /4) ---
  A =   0.1250 + j  0.0000
  B =   0.1250 + j  0.0000
  C =   0.1250 + j  0.0000
  D =   0.1250 + j  0.0000
---------------------------------------------------------
Test Case 3:
  --- Inputs ---
  a =   0.5000
  b =  -0.5000
  c =   0.5000
  d =  -0.5000

  --- Expected Hardware Output (Scaled by /4) ---
  A =   0.0000 + j  0.0000
  B =   0.0000 + j  0.0000
  C =   0.5000 + j  0.0000
  D =   0.0000 + j  0.0000
---------------------------------------------------------
